In [1]:
!pip install -q transformers datasets sentencepiece accelerate evaluate optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 43.1 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import optuna
import gc
from pathlib import Path
from sklearn.model_selection import train_test_split,KFold
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5Tokenizer,
    T5ForConditionalGeneration,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()

Device: cuda


In [3]:
from google.colab import drive
drive.mount("/content/drive")

CSV_PATH = "/content/drive/MyDrive/ASQP/data_processado/aug_sr.csv"

df = pd.read_csv(CSV_PATH)

TEXT_COL = "text"
TARGET_COL = "target"

if "input" not in df.columns:
    df["input"] = "extrair quadruplas: " + df[TEXT_COL].astype(str)

df = df.dropna(subset=["input", TARGET_COL]).copy()

df["input"] = df["input"].astype(str)
df[TARGET_COL] = df[TARGET_COL].astype(str)

print("Total:", len(df))
display(df[["input", TARGET_COL]].head())

Mounted at /content/drive
Total: 2190


,input,target
0,extrair quadruplas:apesar das qualificações na...,[A] atendentes [C] service [O] atenciosos [P] ...
1,extrair quádruplas: estive por dia-a-dia nesse...,[A] quarto [C] structure [O] pequeno [P] neg [...
2,extrair quádruplas: localização sensacional de...,[A] localização [C] location [O] sensacional [...
3,extrair quadruplas:viagem em familia em que se...,[A] localização [C] location [O] excelente [P]...
4,extrair quádruplas: ao inferir ao albergue a r...,[A] recepção [C] service [O] excelente [P] pos...


# Hyperparams Search - Raw x MT5

In [4]:
def tokenize_batch(batch):
  model_inputs = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False
  )

  labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False
  )

  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [ ]:
MODEL_NAME = "google/mt5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]
tokenizer.add_tokens(SPECIAL_TOKENS)

print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer carregado.
Tamanho do vocabulário: 250105


In [ ]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 382
MAX_TARGET_LEN: 152


In [5]:
def clean_pred(text):
  if not isinstance(text, str):
    return ""

  text = text.replace("<extra_id_0>", "")
  text = text.replace("<extra_id_1>", "")
  text = text.replace("<pad>", "")
  text = text.replace("</s>", "")
  text = " ".join(text.split())

  return text.strip()


def parse_target_string(text):
  if not isinstance(text, str):
    return set()

  text = clean_pred(text)

  quads = []

  for part in text.split("[SSEP]"):
      part = part.strip()

      if not part:
        continue

      try:
        a = part.split("[A]")[1].split("[C]")[0].strip()
        c = part.split("[C]")[1].split("[O]")[0].strip()
        o = part.split("[O]")[1].split("[P]")[0].strip()
        p = part.split("[P]")[1].strip()

        quads.append((a, c, o, p))

      except Exception:
        continue

  return set(quads)


def evaluate_asqp(golds, preds):
  total_correct = 0
  total_pred = 0
  total_gold = 0

  for gold, pred in zip(golds, preds):
    gold_set = parse_target_string(gold)
    pred_set = parse_target_string(pred)

    total_correct += len(gold_set & pred_set)
    total_pred += len(pred_set)
    total_gold += len(gold_set)

  precision = total_correct / total_pred if total_pred else 0
  recall = total_correct / total_gold if total_gold else 0

  f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0
  )

  return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "correct": total_correct,
        "pred_total": total_pred,
        "gold_total": total_gold
    }

In [6]:
def generate_predictions(model, tokenizer, texts):
  model.eval()
  preds = []

  for text in texts:
    inputs = tokenizer(
          text,
          return_tensors="pt",
          max_length=MAX_INPUT_LEN,
          truncation=True
    ).to(model.device)

    with torch.no_grad():
      outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                early_stopping=True)

      pred = tokenizer.decode(outputs[0], skip_special_tokens=False)
      pred = clean_pred(pred)

      preds.append(pred)

  return preds

In [7]:
def objective(trial):
  clear_memory()

  learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        5e-4,
        log=True
  )

  num_train_epochs = 6

  weight_decay = trial.suggest_categorical(
        "weight_decay",
        [0.0, 0.001, 0.01]
  )

  warmup_ratio = trial.suggest_categorical(
        "warmup_ratio",
        [0.0, 0.05, 0.1]
  )

  kf = KFold(
        n_splits=3,
        shuffle=True,
        random_state=SEED
  )

  fold_f1s = []

  for fold, (train_idx, val_idx) in enumerate(kf.split(df), start=1):
      print(f"\nTrial {trial.number} | Fold {fold}/3")

      train_fold = df.iloc[train_idx].reset_index(drop=True)
      val_fold = df.iloc[val_idx].reset_index(drop=True)

      train_dataset = Dataset.from_pandas(train_fold)
      val_dataset = Dataset.from_pandas(val_fold)

      train_tokenized = train_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=train_dataset.column_names
      )

      val_tokenized = val_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=val_dataset.column_names
      )

      if MODEL_NAME == "unicamp-dl/ptt5-base-portuguese-vocab":

        model = T5ForConditionalGeneration.from_pretrained("unicamp-dl/ptt5-base-portuguese-vocab")
      else:

        model = AutoModelForSeq2SeqLM.from_pretrained(
              MODEL_NAME,
              tie_word_embeddings=False
        )

      model.resize_token_embeddings(len(tokenizer))
      model.to(DEVICE)

      data_collator = DataCollatorForSeq2Seq(
          tokenizer=tokenizer,
          model=model,
          label_pad_token_id=-100
      )

      fold_out_dir = f"/content/optuna_trial_{trial.number}_fold_{fold}"

      training_args = Seq2SeqTrainingArguments(
            output_dir=fold_out_dir,

            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,

            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps= 1,

            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,

            predict_with_generate=True,
            generation_max_length=MAX_TARGET_LEN,

            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",

            fp16=False,
            report_to="none",

            seed=SEED,
            data_seed=SEED
      )

      trainer = Seq2SeqTrainer(
            model=model,
            args=training_args,
            train_dataset=train_tokenized,
            eval_dataset=val_tokenized,
            data_collator=data_collator
      )

      trainer.train()

      preds = generate_predictions(
            model=model,
            tokenizer=tokenizer,
            texts=val_fold["input"].tolist()
      )

      golds = val_fold["target"].tolist()

      metrics = evaluate_asqp(golds, preds)
      fold_f1 = metrics["f1"]

      print("Fold metrics:", metrics)

      fold_f1s.append(fold_f1)

      del model
      del trainer
      del train_tokenized
      del val_tokenized
      clear_memory()

  mean_f1 = float(np.mean(fold_f1s))

  print(f"\nTrial {trial.number} | Mean CV F1:", mean_f1)

  return mean_f1

In [ ]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-29 04:04:48,078] A new study created in memory with name: no-name-f37ade0b-880b-4c40-9ef2-b3fc4d4a25c7



Trial 0 | Fold 1/3


Map:   0%|          | 0/1460 [00:00<?, ? examples/s]

Map:   0%|          | 0/730 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,4.393854,0.468542
2,0.531150,0.330857
3,0.381916,0.272620
4,0.295547,0.236678
5,0.236717,0.219398
6,0.208180,0.207100


Fold metrics: {'precision': 0.7317512766596576, 'recall': 0.710411198600175, 'f1': 0.7209233501035809, 'correct': 2436, 'pred_total': 3329, 'gold_total': 3429}


[I 2026-06-29 12:23:06,791] Trial 5 finished with value: 0.71777851351758 and parameters: {'learning_rate': 0.00020022156269315273, 'weight_decay': 0.0, 'warmup_ratio': 0.0}. Best is trial 1 with value: 0.763782810159605.



Trial 5 | Mean CV F1: 0.71777851351758
Melhores hiperparâmetros:
{'learning_rate': 0.00037755974095174426, 'weight_decay': 0.01, 'warmup_ratio': 0.1}
Melhor F1 médio 3-fold:
0.763782810159605


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_sr0.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)

,number,value,datetime_start,datetime_complete,duration,params_learning_rate,params_warmup_ratio,params_weight_decay,state
0,0,0.526128,2026-06-29 04:04:48.079741,2026-06-29 05:39:07.211353,0 days 01:34:19.131612,0.000483,0.10,0.00,COMPLETE
1,1,0.763783,2026-06-29 05:39:07.212378,2026-06-29 06:49:26.471279,0 days 01:10:19.258901,0.000378,0.10,0.01,COMPLETE
2,2,0.719914,2026-06-29 06:49:26.472284,2026-06-29 08:00:50.100373,0 days 01:11:23.628089,0.000205,0.10,0.01,COMPLETE
3,3,0.292133,2026-06-29 08:00:50.101475,2026-06-29 09:56:43.922966,0 days 01:55:53.821491,0.000381,0.05,0.00,COMPLETE
4,4,0.672864,2026-06-29 09:56:43.924075,2026-06-29 11:12:08.814085,0 days 01:15:24.890010,0.000121,0.10,0.00,COMPLETE
5,5,0.717779,2026-06-29 11:12:08.815087,2026-06-29 12:23:06.790992,0 days 01:10:57.975905,0.000200,0.00,0.00,COMPLETE


Resultados salvos em: /content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_mt5_sr0.csv


# Hyperparams Search - Raw x PTT5

In [9]:
MODEL_NAME = "unicamp-dl/ptt5-base-portuguese-vocab"

SPECIAL_TOKENS = ["[A]", "[C]", "[O]", "[P]", "[SSEP]"]

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_tokens(SPECIAL_TOKENS)


print("Tokenizer carregado.")
print("Tamanho do vocabulário:", len(tokenizer))

Tokenizer carregado.
Tamanho do vocabulário: 32105


In [10]:
input_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["input"]
]

target_lengths = [
    len(tokenizer(x)["input_ids"])
    for x in df["target"]
]

MAX_INPUT_LEN = int(np.percentile(input_lengths, 99))
MAX_TARGET_LEN = int(np.percentile(target_lengths, 99))

MAX_INPUT_LEN = max(MAX_INPUT_LEN, 64)
MAX_TARGET_LEN = max(MAX_TARGET_LEN, 64)

print("MAX_INPUT_LEN:", MAX_INPUT_LEN)
print("MAX_TARGET_LEN:", MAX_TARGET_LEN)

MAX_INPUT_LEN: 308
MAX_TARGET_LEN: 171


In [11]:
study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=6
)

print("Melhores hiperparâmetros:")
print(study.best_params)

print("Melhor F1 médio 3-fold:")
print(study.best_value)

[I 2026-06-29 14:55:47,793] A new study created in memory with name: no-name-95fcd323-bfca-48f7-b80b-b230596110ab



Trial 0 | Fold 1/3


Map:   0%|          | 0/1460 [00:00<?, ? examples/s]

Map:   0%|          | 0/730 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,4.376010,0.848961
2,0.624934,0.354720
3,0.315114,0.263877
4,0.197324,0.189781
5,0.131951,0.169014
6,0.095147,0.163950


Fold metrics: {'precision': 0.8194902548725638, 'recall': 0.7970253718285214, 'f1': 0.8081017149615611, 'correct': 2733, 'pred_total': 3335, 'gold_total': 3429}


[I 2026-06-29 21:44:59,729] Trial 5 finished with value: 0.8014672083574376 and parameters: {'learning_rate': 0.00045105123614691615, 'weight_decay': 0.0, 'warmup_ratio': 0.1}. Best is trial 5 with value: 0.8014672083574376.



Trial 5 | Mean CV F1: 0.8014672083574376
Melhores hiperparâmetros:
{'learning_rate': 0.00045105123614691615, 'weight_decay': 0.0, 'warmup_ratio': 0.1}
Melhor F1 médio 3-fold:
0.8014672083574376


In [ ]:
optuna_results = study.trials_dataframe()

OPTUNA_PATH = "/content/drive/MyDrive/ASQP/saida_asqp/optuna_3fold_ptt5_sr0.csv"

optuna_results.to_csv(OPTUNA_PATH, index=False)

display(optuna_results)

print("Resultados salvos em:", OPTUNA_PATH)